# Overpower

This notebook computes [overpower](https://chunithm.org/intermediate/rating/#overpower-op) using scores from [Kamaitachi](https://kamai.tachi.ac/). To get started, make a copy of this notebook, fill in your Kamaitachi username and select a track list in the boxes below, then click "Run all" above. To refresh, just click it again. You can use the table of contents on the left to browse.

Contact @kyleforkbomb on Discord if you:
- run into any issues running the notebook
- notice any bugs (wrong numbers, wrong track list, etc.)
- want a new table/graph/track list/whatever

## Notes
- This notebook computes overpower using the system introduced in LUMINOUS PLUS, i.e. per-song instead of per-chart.
- This notebook does not track chart constant changes over time, as sourcing this data is hard and it doesn't provide a lot of value.
- Since this notebook computes overpower using beerpsi's chuni penguin data, the chart constants will mostly track the international version.
- If you import to Kamaitachi from a source that does not provide timestamps (notably, official servers 😞), then the time the score was imported to Kamaitachi will be used instead. Depending on how often you import, this may mess up the "Overpower Over Time" chart, but the other features should still work fine.
- You could edit the cells in this section if you
  - want a different track list
  - want a score source other than Kamaitachi


## Acknowledgements
- tluo5458, for making the original [OP tracker](https://docs.google.com/preadsheets/d/1Rfkwi99HT8pXisWP71xUWrPQz4HOghD-dCpOttrrEU8/edit?usp=sharing) sheet this notebook is heavily based on
- beerpsi, for maintaining and serving the [song list](https://chunithm.beerpsi.cc/songs)

In [ ]:
# @title Fetch Scores

import json
import re
import urllib.parse
from pathlib import Path

import polars as pl
import requests
from IPython.display import Javascript

pl.Config.set_tbl_rows(50)
pl.Config.set_tbl_hide_column_data_types(True)
pl.Config.set_tbl_hide_dataframe_shape(True)


# hahahahahahahaha... https://github.com/pola-rs/polars/issues/18102
if "real_get_fmt" not in globals():
  real_get_fmt = pl.polars.PySeries.get_fmt
  def get_fmt(self: pl.polars.PySeries, index: int, str_len_limit: int) -> str:
    fmt = real_get_fmt(self, index, str_len_limit)
    if self.dtype() == pl.String:
      return re.sub(r'(^"|"$)', "", fmt, count=2)
    return fmt
  pl.polars.PySeries.get_fmt = get_fmt

def set_iframe_height():
  display(Javascript('''google.colab.output.setIframeHeight(0, true, {maxHeight: 1000000})'''))
get_ipython().events.register('pre_run_cell', set_iframe_height)

# -------------------- get scores --------------------

username = "Lolergags" # @param {"type":"string","placeholder":"kfb"}

print("Fetching scores...")

r = requests.get(f"https://kamai.tachi.ac/api/v1/users/{urllib.parse.quote(username, safe='')}/games/chunithm/Single/scores/all")
scores_json = r.json()["body"]

print(f"Fetched {len(scores_json['scores'])} scores from Kamaitachi")

kt_diff_map = {
  "BASIC": "BAS",
  "ADVANCED": "ADV",
  "EXPERT": "EXP",
  "MASTER": "MAS",
  "ULTIMA": "ULT",
}

chart_difficulty = {
  chart["chartID"]: kt_diff_map[chart["difficulty"]]
  for chart in scores_json["charts"]
}

scores_df = pl.from_dicts(
  {
    "time": score["timeAchieved"] or score["timeAdded"],
    "id": score["songID"],
    "difficulty": chart_difficulty[score["chartID"]],
    "score": score["scoreData"]["score"],
    "lamp": score["scoreData"]["noteLamp"],
    "service": score["service"],
    "import": score["importType"],
  } | score["scoreData"]["judgements"]
  for score in scores_json["scores"]
).with_columns(
  pl.from_epoch("time", time_unit="ms"),
  lamp_bonus=pl.col("lamp").replace_strict(
    ["NONE", "FULL COMBO", "ALL JUSTICE", "ALL JUSTICE CRITICAL"],
    [0, 500, 1000, 1250],
  ),
)

Fetching scores...
Fetched 6769 scores from Kamaitachi


In [ ]:
# @title Calculate Overpower
# @markdown If you change these settings, you can highlight this cell and press
# @markdown Ctrl+F10 to skip fetching your scores from Kamaitachi and re-run
# @markdown everything else.

def fetch_songs_json():
  p = Path("songs.json")
  if p.is_file():
    print("Using cached songs list")
    return json.loads(p.read_text())
  print("Fetching songs list...")
  r = requests.get("https://chunithm.beerpsi.cc/songs")
  p.write_text(r.text)
  return r.json()

charts_df = pl.from_dicts(
  {
    "id": song["id"],
    "title": song["title"],
    "artist": song["artist"],
    "genre": song["genre"],
    "version": song["version"],
    "difficulty": chart["difficulty"],
    "const": chart["const"],
    "added_version": chart["version"] or song["version"],
    "available_jp": song["availability"]["jp"],
    "available_intl": song["availability"]["intl"],
    "notes": chart["notecounts"]["total"],
  }
  for song in fetch_songs_json()
  for chart in song["charts"]
)

print(f"Loaded {charts_df.height} charts")

# -------------------- build track lists --------------------

versions = [
  "CHUNITHM", "CHUNITHM PLUS", "AIR", "AIR PLUS", "STAR", "STAR PLUS", "AMAZON",
  "AMAZON PLUS", "CRYSTAL", "CRYSTAL PLUS", "PARADISE", "PARADISE LOST", "NEW",
  "NEW PLUS", "SUN", "SUN PLUS", "LUMINOUS", "LUMINOUS PLUS", "VERSE",
  "X-VERSE",
]

versions_df = pl.DataFrame(
  {
    "version_name": versions,
    "version_number": range(len(versions)),
  }
)

tutorials = {50, 81}

# not in AOMN
aomn_remove = {0, 5, 9, 10, 12, 14, 17, 28, 34, 36, 39, 42, 43, 54, 55, 56, 57, 58, 60, 78, 84, 85, 86, 87, 109, 110, 111, 112, 116, 124, 125, 126, 129, 130, 154, 155, 176, 182, 184, 185, 206, 207, 209, 214, 215, 231, 235, 238, 243, 247, 255, 269, 296, 299, 308, 309, 311, 313, 315, 344, 345, 348, 349, 350, 351, 352, 353, 355, 356, 357, 358, 359, 360, 410, 419, 420, 422, 424, 454, 473, 495, 501, 509, 522, 523, 529, 530, 531, 537, 541, 542, 544, 545, 546, 579, 580, 581, 582, 591, 609, 610, 611, 612, 613, 620, 621, 622, 623, 624, 639, 640, 642, 644, 645, 646, 647, 648, 649, 650, 651, 652, 682, 724, 725, 726, 727, 728, 769, 770, 778, 794, 808}

aomn = charts_df.filter(
  ~(pl.col("id").is_in(tutorials | aomn_remove)) & (pl.col("difficulty") != "WE")
)

jp = charts_df.filter(
  ~(pl.col("id").is_in(tutorials))
  & (pl.col("difficulty") != "WE")
  & pl.col("available_jp")
)

intl = charts_df.filter(
  ~(pl.col("id").is_in(tutorials))
  & (pl.col("difficulty") != "WE")
  & pl.col("available_intl")
)

track_list = aomn # @param ["aomn","jp","intl"] {"type":"raw"}
version = "VERSE" # @param ["VERSE","LUMINOUS PLUS","LUMINOUS"] {"type":"string"}
track_list_filtered = (
    track_list.join(versions_df, left_on="added_version", right_on="version_name", how="inner")
    .filter(pl.col("version_number") <= versions.index(version))
  ).drop("version_number", "added_version")

# -------------------- compute overpower --------------------

level_base = pl.col("const") * 10_000
overpower_base_df = (
  scores_df.join(charts_df, on=["id", "difficulty"])
  .with_columns(
    overpower_base=(
      pl.when(pl.col("score") >= 1_007_500)
      .then(level_base + 20_000 + (pl.col("score") - 1_007_500) * 3)
      .otherwise(
        pl.when(pl.col("score") >= 1_005_000)
        .then(level_base + 15_000 + (pl.col("score") - 1_005_000) * 2)
        .otherwise(
          pl.when(pl.col("score") >= 1_000_000)
          .then(level_base + 10_000 + (pl.col("score") - 1_000_000))
          .otherwise(
            pl.when(pl.col("score") >= 975_000)
            .then(level_base + (pl.col("score") - 975_000) * 2 / 5)
            .otherwise(
              pl.when(pl.col("score") >= 900_000)
              .then(
                level_base
                - 50_000
                + (pl.col("score") - 900_000) * 2 / 3
              )
              .otherwise(
                pl.when(pl.col("score") >= 800_000)
                .then(
                  (level_base - 50_000) / 2
                  + (
                    (pl.col("score") - 800_000)
                    * ((level_base - 50_000) / 2)
                  )
                  / 100_000
                )
                .otherwise(
                    pl.when(pl.col("score") >= 500_000)
                    .then(
                      (
                        ((level_base - 50_000) / 2)
                        * (pl.col("score") - 500_000)
                      )
                      / 300_000
                    )
                    .otherwise(0)
                )
              )
            )
          )
        )
      )
      / 2
    )
    .floor()
    .cast(pl.Int64),
  )
  .with_columns(
    overpower_base=pl.when(pl.col("score") >= 975_000)
    .then((pl.col("overpower_base") / 5).floor().cast(pl.Int64) * 5)
    .otherwise((pl.col("overpower_base") / 50).floor().cast(pl.Int64) * 50)
  )
)

lamp_bonus_df = scores_df.group_by("id", "difficulty").agg(pl.max("lamp_bonus"))
overpower_df = (
  overpower_base_df.group_by("id", "difficulty")
  .agg(pl.max("overpower_base"))
  .join(lamp_bonus_df, on=["id", "difficulty"])
  .with_columns(overpower=pl.col("overpower_base") + pl.col("lamp_bonus"))
)

best_scores = scores_df.group_by("id", "difficulty").agg(
  pl.max("score"),
  (pl.col("lamp") == "FULL COMBO").any().alias("fc"),
  (pl.col("lamp") == "ALL JUSTICE").any().alias("aj"),
  (pl.col("lamp") == "ALL JUSTICE CRITICAL").any().alias("ajc"),
)

combo_df = (
  track_list_filtered.join(best_scores, on=["id", "difficulty"], how="left")
  .join(overpower_df, on=["id", "difficulty"], how="left")
  .with_columns(
    pl.col("score").fill_null(0),
    pl.col("overpower").fill_null(0),
    (pl.col("const") * 5 + 15).alias("max_overpower"),
    (pl.col("fc") | pl.col("aj") | pl.col("ajc")).fill_null(False),
    (pl.col("aj") | pl.col("ajc")).fill_null(False),
    pl.col("ajc").fill_null(False),
  )
)

combo_best_difficulty_df = combo_df.group_by("id").agg(
  pl.all().exclude("fc", "aj", "ajc", "max_overpower").sort_by("overpower").last(),
  pl.col("fc", "aj", "ajc").sort_by("const").last(),
  pl.max("max_overpower"),
)

op = (combo_df.group_by("id").agg(pl.max("overpower"))["overpower"].sum()) // 10 / 100
max_op = combo_df.group_by("id").agg(pl.max("max_overpower"))["max_overpower"].sum()

<IPython.core.display.Javascript object>

Fetching songs list...
Loaded 7909 charts


In [ ]:
# @title OP tracker import
# @markdown If the OP tracker sheet import script is not working, you can paste
# @markdown this into the "Raw Import" tab instead. Make sure to click the copy
# @markdown button instead of copying it manually.

scores_df.group_by("id", "difficulty").agg(
    pl.max("score"),
    fc=pl.col("lamp").is_in({"FULL COMBO", "ALL JUSTICE", "ALL JUSTICE CRITICAL"}).any(),
    aj=pl.col("lamp").is_in({"ALL JUSTICE", "ALL JUSTICE CRITICAL"}).any(),
).join(charts_df, on=["id", "difficulty"]).select(
    Song="title",
    Diff="difficulty",
    Score="score",
    FC="fc",
    AJ="aj",
).write_csv(separator="\t")

<IPython.core.display.Javascript object>

'Song\tDiff\tScore\tFC\tAJ\n天体観測\tMAS\t1009991\ttrue\ttrue\nB.B.K.K.B.K.K.\tMAS\t1006041\tfalse\tfalse\nB.B.K.K.B.K.K.\tULT\t1008154\ttrue\tfalse\nScatman (Ski Ba Bop Ba Dop Bop)\tMAS\t1009904\ttrue\ttrue\nReach For The Stars\tMAS\t1007645\tfalse\tfalse\nReach For The Stars\tULT\t1001152\tfalse\tfalse\n初音ミクの消失\tMAS\t1007240\tfalse\tfalse\n情熱大陸\tMAS\t1009232\ttrue\tfalse\nAll I Want\tMAS\t1009679\ttrue\ttrue\n紅蓮の弓矢\tMAS\t1009810\ttrue\ttrue\nコネクト\tMAS\t1009953\ttrue\ttrue\n空色デイズ\tMAS\t1009879\ttrue\ttrue\n千本桜\tMAS\t1007198\tfalse\tfalse\n千本桜\tULT\t1007538\tfalse\tfalse\nDRAGONLADY\tMAS\t1007205\tfalse\tfalse\ntaboo tears you up\tMAS\t1009197\ttrue\tfalse\ntaboo tears you up\tULT\t1005088\tfalse\tfalse\nナイト・オブ・ナイツ\tMAS\t1009921\ttrue\ttrue\n一触即発☆禅ガール\tMAS\t1006474\tfalse\tfalse\nドーナツホール\tMAS\t1008614\ttrue\tfalse\nタイガーランペイジ\tMAS\t1008534\tfalse\tfalse\nタイガーランペイジ\tULT\t1009142\ttrue\tfalse\nPursuing My True Self\tMAS\t1009945\ttrue\ttrue\nBlue Noise\tMAS\t995455\tfalse\tfalse\n亡國覚醒カタルシス\t

# Aggregations

In [ ]:
# @title All Overpower

print(f"{op} / {max_op} ({int(op / max_op * 10000) / 100}%)\n")

overpower_over_time = (
  overpower_base_df.join(track_list_filtered, on=["id", "difficulty"])
  .sort("time")
  .with_columns(
    overpower=(
      pl.col("overpower_base").cum_max() + pl.col("lamp_bonus").cum_max()
    ).over("id", "difficulty")
  )
  .with_columns(
    overpower_gain=(
      pl.col("overpower")
      - (pl.col("overpower").cum_max().shift(1, fill_value=0).over("id"))
    ).clip(0)
  )
  .select(
    "time",
    pl.col("overpower_gain").cum_sum().alias("overpower") // 10 / 100,
  )
)

overpower_over_time.group_by_dynamic("time", every="1d").agg(
  pl.last("overpower")
).plot.line(x="time", y="overpower").properties(
  title="Overpower Over Time", width=800, height=400
)

<IPython.core.display.Javascript object>

124564.08 / 131427.5 (94.77%)



alt.Chart(...)

In [ ]:
# @title Overpower by Level

def overpower_by(df: pl.DataFrame, expr: pl.Expr) -> pl.DataFrame:
  return (
    df.group_by(expr)
    .agg(
      pl.len().alias("#"),
      (pl.col("score") > 0).sum().alias("played"),
      pl.sum("fc"),
      pl.sum("aj"),
      pl.sum("ajc"),
      ((pl.col("overpower")).sum() // 10 / 100).round(2).alias("overpower"),
      pl.sum("max_overpower"),
      ((pl.col("overpower") > 0) * pl.col("max_overpower"))
      .sum()
      .alias("filtered_max_overpower"),
    )
    .with_columns(
      (pl.col("overpower") / pl.col("max_overpower") * 100)
      .round(3)
      .alias("overpower_percent"),
      (pl.col("overpower") / pl.col("filtered_max_overpower") * 100)
      .round(3)
      .alias("filtered_overpower_percent"),
    )
    .select(
      pl.nth(0),
      pl.col("#"),
      pl.col("played").alias("Played"),
      pl.col("fc").alias("FC"),
      pl.col("aj").alias("AJ"),
      pl.col("ajc").alias("AJC"),
      pl.col("overpower").alias("Overpower"),
      pl.col("max_overpower").alias("Max Overpower"),
      pl.col("overpower_percent").alias("Overpower %"),
      pl.col("filtered_max_overpower").alias("Filtered Max Overpower"),
      pl.col("filtered_overpower_percent").alias("Filtered Overpower %"),
    )
  )


overpower_by(combo_df, (pl.col("const").alias("Level") * 2).floor() / 2).sort(
  "Level", descending=True
)

<IPython.core.display.Javascript object>

Level,#,Played,FC,AJ,AJC,Overpower,Max Overpower,Overpower %,Filtered Max Overpower,Filtered Overpower %
15.5,22,22,0,0,0,1801.44,2042.5,88.198,2042.5,88.198
15.0,134,132,1,0,0,10795.83,12147.5,88.873,11966.5,90.217
14.5,189,189,10,1,0,15362.01,16739.5,91.771,16739.5,91.771
14.0,308,307,58,8,0,24717.58,26470.5,93.378,26385.0,93.68
13.5,334,317,79,29,0,24940.19,27893.0,89.414,26477.5,94.194
13.0,268,218,94,41,0,16861.64,21696.5,77.716,17639.5,95.59
12.5,294,240,115,52,0,18111.68,23066.0,78.521,18841.5,96.127
12.0,228,157,140,119,2,11797.14,17334.0,68.058,11941.0,98.795
11.5,195,141,141,141,0,10324.69,14341.5,71.992,10378.5,99.482
11.0,155,54,54,54,0,3831.35,11008.5,34.804,3852.0,99.464


In [ ]:
# @title Overpower by Genre

overpower_by(combo_best_difficulty_df, pl.col("genre").alias("Genre"))

<IPython.core.display.Javascript object>

Genre,#,Played,FC,AJ,AJC,Overpower,Max Overpower,Overpower %,Filtered Max Overpower,Filtered Overpower %
イロドリミドリ,101,101,45,23,0,7761.88,8109.5,95.713,8109.5,95.713
POPS & ANIME,240,240,163,134,0,18207.74,18711.5,97.308,18711.5,97.308
VARIETY,240,240,47,19,0,19135.11,20600.0,92.889,20600.0,92.889
ORIGINAL,394,394,89,47,0,31123.53,33426.5,93.11,33426.5,93.11
niconico,303,303,177,121,1,23343.58,24215.0,96.401,24215.0,96.401
東方Project,146,146,66,36,1,11285.85,11810.0,95.562,11810.0,95.562
ゲキマイ,173,173,49,32,0,13706.37,14555.0,94.169,14555.0,94.169


In [ ]:
# @title Overpower by Version

overpower_by(combo_best_difficulty_df, pl.col("version").alias("Version")).join(
  versions_df, left_on="Version", right_on="version_name", how="left"
).sort("version_number").drop("version_number")

<IPython.core.display.Javascript object>

Version,#,Played,FC,AJ,AJC,Overpower,Max Overpower,Overpower %,Filtered Max Overpower,Filtered Overpower %
CHUNITHM,94,94,32,21,0,7227.61,7729.0,93.513,7729.0,93.513
CHUNITHM PLUS,47,47,15,6,0,3618.73,3882.0,93.218,3882.0,93.218
AIR,78,78,32,22,0,6001.01,6356.5,94.407,6356.5,94.407
AIR PLUS,70,70,31,21,0,5379.53,5649.5,95.221,5649.5,95.221
STAR,77,77,33,20,0,5981.34,6290.5,95.085,6290.5,95.085
STAR PLUS,73,73,31,16,0,5614.81,5948.5,94.39,5948.5,94.39
AMAZON,65,65,25,16,1,5062.14,5343.0,94.743,5343.0,94.743
AMAZON PLUS,73,73,28,21,0,5681.23,5990.5,94.837,5990.5,94.837
CRYSTAL,90,90,38,28,0,6937.91,7284.0,95.249,7284.0,95.249
CRYSTAL PLUS,86,86,38,25,0,6652.2,6983.5,95.256,6983.5,95.256


# Possession

In [ ]:
# @title Lamps
# @markdown If `cascade` is checked, then AJC will count as AJ+FC and AJ will
# @markdown count as FC. This matches the OP Tracker sheet's lamp table.
cascade = False # @param {"type":"boolean"}


labels = ["Played", "FC", "AJ", "AJC"]
difficulties = pl.DataFrame(
    {
        "difficulty": ["BAS", "ADV", "EXP", "MAS", "ULT"],
    }
)

filters = [
    pl.col("score") > 0,
    pl.col("fc"),
    pl.col("aj"),
    pl.col("ajc"),
]

if not cascade:
    filters[1] &= ~pl.col("aj") & ~pl.col("ajc")
    filters[2] &= ~pl.col("ajc")

pl.concat(
    [
        difficulties.join(
            combo_df.filter(expr).group_by("difficulty").agg(pl.len()), on="difficulty", how="left"
        )
        .fill_null(0)
        .transpose(column_names="difficulty")
        for expr in filters
    ]
).select(
    pl.Series(name="", values=labels),
    pl.col("MAS"),
    pl.col("ULT"),
    (pl.col("MAS") + pl.col("ULT")).alias("Total"),
)

<IPython.core.display.Javascript object>

,MAS,ULT,Total
Played,1597,83,1680
FC,222,10,232
AJ,429,4,433
AJC,2,0,2


In [ ]:
# @title Possession Thresholds
import polars as pl

pos_reqs = [
  ("Silver", 0, "S", 975_000),
  ("Gold", 0.975, "S+", 990_000),
  ("Platinum", 0.99, "SS", 1_000_000),
  ("Rainbow", 0.995, "SSS", 1_007_500),
]

pos_thres_df = pl.from_dicts(
  {
    "pos_name": pos_name,
    "op": max_op * op_pct,
    "rank": rank_name,
    "rank_score": rank_score,
    "rank_left": combo_df.filter(pl.col("difficulty").is_in({"MAS", "ULT"}) & (pl.col("score").fill_null(0) < rank_score)).height,
  }
  for pos_name, op_pct, rank_name, rank_score in pos_reqs
)

pos_done = pos_thres_df.filter((pl.col("op") <= op) & (pl.col("rank_left") == 0))
if pos_done.height == 0:
  print("Possession: None\n")
else:
  print(f"Possession: {pos_done['pos_name'].last()}\n")

pos_thres_df.select(
  pl.col("pos_name").alias("Possession"),
  pl.col("op").alias("OP"),
  (pl.col("op") - op).clip(0).alias("OP Left"),
  pl.col("rank").alias("Rank"),
  pl.col("rank_left").alias("Rank Left"),
)

<IPython.core.display.Javascript object>

Possession: None



Possession,OP,OP Left,Rank,Rank Left
Silver,0.0,0.0,S,3
Gold,128141.8125,3577.7325,S+,3
Platinum,130113.225,5549.145,SS,110
Rainbow,130770.3625,6206.2825,SSS,669


In [ ]:
# @title Next Possession Missing Charts

rank_missing_df = pos_thres_df.filter(pl.col("rank_left") != 0)
if rank_missing_df.height == 0:
  print("Everything done :)")
  remaining = combo_df.clear()
else:
  remaining = combo_df.filter(pl.col("difficulty").is_in({"MAS", "ULT"})).filter(
    pl.col("score").fill_null(0) < rank_missing_df["rank_score"][0]
  )
  print(f"{remaining.height} charts left for {rank_missing_df['pos_name'][0]} Possession")

<IPython.core.display.Javascript object>

3 charts left for Silver Possession


In [ ]:
# @title Missing Charts by Version

remaining.select("version", "id", "difficulty").group_by("version").agg(
  (pl.col("difficulty") == "MAS").sum().alias("mas_left"),
  (pl.col("difficulty") == "ULT").sum().alias("ult_left"),
).join(versions_df, left_on="version", right_on="version_name").sort(
  "version_number"
).select(
  pl.col("version").alias("Version"),
  pl.col("mas_left").fill_null(0).alias("MAS Left"),
  pl.col("ult_left").fill_null(0).alias("ULT Left"),
)

<IPython.core.display.Javascript object>

Version,MAS Left,ULT Left
AIR,0,1
AMAZON,0,1
VERSE,0,1


In [ ]:
# @title Missing Charts by Genre

remaining.select(pl.col("genre").alias("Genre"), "id", "difficulty").group_by("Genre").agg(
  (pl.col("difficulty") == "MAS").sum().alias("MAS Left"),
  (pl.col("difficulty") == "ULT").sum().alias("ULT Left"),
)

<IPython.core.display.Javascript object>

Genre,MAS Left,ULT Left
VARIETY,0,2
POPS & ANIME,0,1


In [ ]:
# @title Missing Charts by Constant
# @markdown Top 50 easiest missing charts
remaining.sort("const").select(
  pl.col("title").alias("Title"),
  pl.col("difficulty").alias("Difficulty"),
  pl.col("const").alias("Constant"),
  pl.col("genre").alias("Genre"),
  pl.col("version").alias("Version"),
).head(50)

<IPython.core.display.Javascript object>

Title,Difficulty,Constant,Genre,Version
Campus mode!!,ULT,14.1,POPS & ANIME,VERSE
DataErr0r,ULT,15.1,VARIETY,AIR
Parousia,ULT,15.1,VARIETY,AMAZON


In [ ]:
# @title Possible Overpower Gain
# @markdown Top 50 charts by overpower gain, up to a difficulty
max_const = 14.4 # @param {"type":"number","placeholder":"14.2"}

(
  combo_df.drop("overpower")
  .join(combo_best_difficulty_df.select("id", "overpower"), on="id")
  .with_columns(missing=(pl.col("max_overpower") - (pl.col("overpower") // 10 / 100)))
  .filter(pl.col("missing") > 0)
  .filter(pl.col("const") <= max_const)
  .sort("missing", descending=True)
  .select(
    pl.col("title").alias("Title"),
    pl.col("difficulty").alias("Difficulty"),
    pl.col("const").alias("Constant"),
    pl.col("genre").alias("Genre"),
    pl.col("version").alias("Version"),
    pl.col("overpower").alias("Best Song Overpower") // 10 / 100,
    pl.col("max_overpower").alias("Max Chart Overpower"),
    pl.col("missing").alias("Overpower Gain")
  )
  .head(50)
)

<IPython.core.display.Javascript object>

Title,Difficulty,Constant,Genre,Version,Best Song Overpower,Max Chart Overpower,Overpower Gain
オススメ☆♂♀☆でぃすとぴあ,MAS,14.4,ORIGINAL,AIR PLUS,77.07,87.0,9.93
きゅうりバーにダイブ,MAS,14.3,東方Project,STAR PLUS,76.59,86.5,9.91
音弾超人ゴリライザー,MAS,13.9,VARIETY,CRYSTAL,74.7,84.5,9.8
SILENT BLUE,MAS,14.2,ゲキマイ,LUMINOUS,76.24,86.0,9.76
Jack-the-Ripper◆,MAS,14.2,VARIETY,CHUNITHM PLUS,76.25,86.0,9.75
任せてJC陰陽師☆八雲ちゃん,MAS,13.3,ORIGINAL,SUN,71.77,81.5,9.73
Alice's Suitcase,MAS,13.3,VARIETY,SUN,71.78,81.5,9.72
Iudicium “Apocalypsis Mix”,MAS,14.4,ゲキマイ,NEW,77.31,87.0,9.69
GEMINI -C-,MAS,14.2,ORIGINAL,CHUNITHM PLUS,76.36,86.0,9.64
felys -final remix-,MAS,14.4,VARIETY,CRYSTAL PLUS,77.39,87.0,9.61


In [ ]:
# @title Almost All Justice
# @markdown Top 50 MAS/ULT charts by difficulty, with lowest attack+miss between
# @markdown 1 and `threshold`
# @markdown
# @markdown Note that these may not be your highest scores on these charts

# make sure all the notes were actually played
full_played = track_list_filtered.join(scores_df, on=["id", "difficulty"]).filter(
  pl.col("notes") == pl.sum_horizontal("jcrit", "justice", "attack", "miss"),
  pl.col("difficulty").is_in({"MAS", "ULT"}),
)

threshold = 1 # @param {"type":"integer","placeholder":"1"}

almost_aj = full_played.with_columns(mistakes=pl.sum_horizontal("attack", "miss")).group_by("id", "difficulty").agg(
  pl.first("title", "artist", "genre", "version", "const"),
  pl.col("score", "jcrit", "justice", "attack", "miss", "mistakes").sort_by("mistakes", "miss").first(),
).filter(pl.col("mistakes").is_between(1, threshold)).sort("const").select(
  Title="title",
  Difficulty="difficulty",
  Constant="const",
  Genre="genre",
  Version="version",
  Score="score",
  JC="jcrit",
  J="justice",
  A="attack",
  M="miss",
)

print(f"{almost_aj.height} total")
almost_aj.head(50)

<IPython.core.display.Javascript object>

117 total


Title,Difficulty,Constant,Genre,Version,Score,JC,J,A,M
超常マイマイン,MAS,12.0,ゲキマイ,SUN,1009402,1987,19,0,1
どこにもいかない,MAS,12.0,イロドリミドリ,LUMINOUS,1009463,1267,18,1,0
Black'n White JAMMIN' CATS,MAS,12.2,イロドリミドリ,AMAZON,1009256,1519,13,0,1
ハート・ビート,MAS,12.2,イロドリミドリ,CHUNITHM PLUS,1009087,1395,29,0,1
このふざけた素晴らしき世界は、僕の為にある,MAS,12.2,niconico,CHUNITHM PLUS,1009245,1632,24,0,1
BURST THE GRAVITY,MAS,12.3,POPS & ANIME,CRYSTAL,1009277,1499,8,0,1
帝国少女,MAS,12.3,niconico,NEW,1009515,1386,17,1,0
目覚めRETURNER,MAS,12.3,POPS & ANIME,CRYSTAL,1009592,1440,8,1,0
テリトリーバトル,MAS,12.4,ORIGINAL,PARADISE,1009233,1438,10,0,1
僕の戦争,MAS,12.5,POPS & ANIME,NEW PLUS,1009455,2203,20,0,1


In [ ]:
# @title Almost Full Combo
# @markdown Top 50 MAS/ULT charts by difficulty, with lowest miss between 1 and
# @markdown `threshold`

threshold = 1 # @param {"type":"integer","placeholder":"1"}

almost_fc = full_played.group_by("id", "difficulty").agg(
  pl.first("title", "artist", "genre", "version", "const"),
  pl.col("score", "jcrit", "justice", "attack", "miss").sort_by("miss").first(),
).filter(pl.col("miss").is_between(1, threshold)).sort("const").select(
  Title="title",
  Difficulty="difficulty",
  Constant="const",
  Genre="genre",
  Version="version",
  Score="score",
  JC="jcrit",
  J="justice",
  A="attack",
  M="miss",
)

print(f"{almost_fc.height} total")
almost_fc.head(50)

<IPython.core.display.Javascript object>

253 total


Title,Difficulty,Constant,Genre,Version,Score,JC,J,A,M
Theme of SeelischTact,MAS,12.0,ORIGINAL,CHUNITHM,1008651,1221,15,1,1
神曲,MAS,12.0,niconico,AIR,1008618,1541,12,2,1
響,MAS,12.0,ORIGINAL,AIR PLUS,1008420,2455,39,5,1
超常マイマイン,MAS,12.0,ゲキマイ,SUN,1008649,1986,17,3,1
FEEL the BEATS,MAS,12.1,ゲキマイ,AMAZON,1008233,1485,11,3,1
Barbed Eye,MAS,12.1,ゲキマイ,AIR PLUS,1008165,1751,21,4,1
brilliant better,MAS,12.1,イロドリミドリ,CHUNITHM,1008543,1508,20,2,1
ハート・ビート,MAS,12.2,イロドリミドリ,CHUNITHM PLUS,1008385,1395,27,2,1
ハッピーエンドをはじめから,MAS,12.2,POPS & ANIME,NEW PLUS,1008825,1860,18,2,1
Counselor,MAS,12.3,ORIGINAL,CHUNITHM,1008665,1241,16,1,1


In [ ]:
# @title Almost All Justice Critical
# @markdown Top 50 MAS/ULT charts by difficulty, with lowest justice+attack+miss
# @markdown between 1 and `threshold`

threshold = 3 # @param {"type":"integer","placeholder":"1"}

almost_ajc = full_played.with_columns(mistakes=pl.sum_horizontal("justice", "attack", "miss")).group_by("id", "difficulty").agg(
  pl.first("title", "artist", "genre", "version", "const"),
  pl.col("score", "jcrit", "justice", "attack", "miss", "mistakes").sort_by("mistakes", "miss", "attack", "justice").first(),
).filter(pl.col("mistakes").is_between(1, threshold)).sort("const").select(
  Title="title",
  Difficulty="difficulty",
  Constant="const",
  Genre="genre",
  Version="version",
  Score="score",
  JC="jcrit",
  J="justice",
  A="attack",
  M="miss",
)

print(f"{almost_ajc.height} total")
almost_ajc.head(50)

<IPython.core.display.Javascript object>

21 total


Title,Difficulty,Constant,Genre,Version,Score,JC,J,A,M
輪舞-revolution,MAS,10.7,POPS & ANIME,CRYSTAL,1009984,1252,2,0,0
歩いていこう！,MAS,11.2,POPS & ANIME,PARADISE,1009992,1265,1,0,0
かっとばせ！未来へ,MAS,11.4,POPS & ANIME,CRYSTAL PLUS,1009985,2014,3,0,0
FRIDAY FRIDAY,MAS,11.4,ORIGINAL,AMAZON,1009985,1333,2,0,0
Ring,MAS,11.5,ORIGINAL,NEW,1009977,1310,3,0,0
magnet,MAS,11.5,niconico,CRYSTAL,1009983,1191,2,0,0
みくみくにしてあげる♪【してやんよ】,MAS,11.5,niconico,AIR,1009982,1137,2,0,0
どうぶつ☆パラダイス,MAS,11.5,ゲキマイ,NEW,1009983,1803,3,0,0
ヒトリシズカ(Arranged: Iceon),MAS,11.9,東方Project,LUMINOUS PLUS,1009976,834,2,0,0
君のせい,MAS,11.9,POPS & ANIME,PARADISE,1009982,1700,3,0,0
